# Login / Logout 로그 SQL 생성 (최근 30일 추가분)
- 로그인 → 로그아웃 쌍으로 생성 (PAIR_COUNT=500 -> 총 1000건)
- 날짜 범위: 2026-05-18 ~ 2026-06-16 (2026-06-17 이전 기준 최근 30일)
- 시간대: 0~23시 완전 랜덤
- user_login_id: user0001 ~ user0100, user_id: 1 ~ 100 매칭
- client_uuid: 세션마다 고유 UUID
- 타임스탬프: login < logout 순서 보장
- 모든 로그는 first_save_history 테이블 하나에 저장됨

In [1]:
import random
import json
import uuid
from datetime import datetime, timedelta

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
# 최근 30일 (2026-06-17 이전 기준 -> 2026-05-18 ~ 2026-06-16)
START_DATE  = datetime(2026, 5, 18, 0, 0, 0)
END_DATE    = datetime(2026, 6, 16, 23, 59, 59)
PAIR_COUNT  = 500   # 생성할 login/logout 쌍 수 (총 1000건)

# 세션 체류 시간: 최소 1분 ~ 최대 120분
SESSION_MIN_SEC = 60
SESSION_MAX_SEC = 60 * 120

In [3]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

In [4]:
rows = []

for _ in range(PAIR_COUNT):
    # 유저 선택 (user_id 1~100, user_login_id 매칭)
    user_id       = random.randint(1, 100)
    user_login_id = f'user{user_id:04d}'

    # 세션마다 고유 UUID
    client_uuid = str(uuid.uuid4())

    # ── LOGIN 타임스탬프 ──
    # event_timestamp 기준으로 먼저 뽑고, history_timestamp = event_ts + 1초 (서버 수신이 더 늦음)
    # 시간대는 0~23시 완전 랜덤 (random_datetime이 초 단위 균등 분포)
    login_event_ts   = random_datetime(START_DATE, END_DATE)
    login_history_ts = login_event_ts + timedelta(seconds=1)

    # ── LOGOUT 타임스탬프 ──
    # 세션 체류 시간만큼 뒤
    session_duration  = random.randint(SESSION_MIN_SEC, SESSION_MAX_SEC)
    logout_event_ts   = login_event_ts + timedelta(seconds=session_duration)
    logout_history_ts = logout_event_ts + timedelta(seconds=1)

    # END_DATE 초과 방지
    if logout_event_ts > END_DATE:
        logout_event_ts   = END_DATE
        logout_history_ts = END_DATE + timedelta(seconds=1)

    # ── LOGIN JSON ──
    login_json = json.dumps({
        'event_name':      'login',
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(login_event_ts)
    }, ensure_ascii=False)

    # ── LOGOUT JSON ──
    logout_json = json.dumps({
        'event_name':      'logout',
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(logout_event_ts)
    }, ensure_ascii=False)

    rows.append((login_history_ts, login_json))
    rows.append((logout_history_ts, logout_json))

print(f'✅ {PAIR_COUNT}쌍 생성 완료 (총 {len(rows)}건 = login {PAIR_COUNT}건 + logout {PAIR_COUNT}건)')

✅ 500쌍 생성 완료 (총 1000건 = login 500건 + logout 500건)


In [5]:
# SQL 생성 및 저장 (first_save_history 테이블 하나로)
lines  = ['INSERT INTO first_save_history (history_timestamp, json_log) VALUES']
values = []

for history_ts, json_log in rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('login_logout_logs_recent30d.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {len(rows)}개 login/logout 로그 SQL 생성 완료 → login_logout_logs_recent30d.sql')

✅ 1000개 login/logout 로그 SQL 생성 완료 → login_logout_logs_recent30d.sql


In [6]:
# ── 미리보기 ──
print('=== LOGIN/LOGOUT SQL (앞 500자) ===')
print(sql[:500])

=== LOGIN/LOGOUT SQL (앞 500자) ===
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2026-06-03 05:19:55.000000', '{"event_name": "login", "user_id": 21, "user_login_id": "user0021", "client_uuid": "3381b7b8-8e73-4c61-ad98-3df16760a3b2", "event_timestamp": "2026-06-03T05:19:54.000+09:00"}'),
  ('2026-06-03 05:23:48.000000', '{"event_name": "logout", "user_id": 21, "user_login_id": "user0021", "client_uuid": "3381b7b8-8e73-4c61-ad98-3df16760a3b2", "event_timestamp": "2026-06-03T05:23:47.000+09:00"}'),
  ('202
